# Agent Router v2 -- reduced-leakage features

**Question:** with ONLY routing-time-visible features, can the router beat always-Codex?

Reduced-leakage design (drops the post-treatment PR body and its derivatives):
- Inputs reduced to lower leakage: cleaned PR title (a task-statement proxy) + repo metadata
  (language, log stars, log forks) + has_issue + task_type. The post-treatment PR body and its
  derivatives (body length, code-block count, stack-trace flag) are dropped. The title is still
  agent-authored, so this is reduced-leakage rather than strictly pre-treatment.
- Title cleaning: agent identity tokens are stripped from the title.
- Flat per-agent cost in both the subgroup table and the sweep. A log(forks) task-size proxy was
  tried but collapses to a coarse binary, since about 74% of repos have 0 forks.
- Ablation: a repo-metadata-only model (no text) as the strict lower bound.

Caveat kept as-is (deferred): direct-method eval, no IPW/CI/time-split; still_open=0.3 label.

In [1]:
import os, re, warnings
import numpy as np, pandas as pd
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error, f1_score
import xgboost as xgb
warnings.filterwarnings("ignore")

DATA_PATH="data/router_dataset.jsonl"
AGENTS=["OpenAI_Codex","Copilot","Devin","Cursor","Claude_Code"]
SUCCESS_MAP={"merged":1.0,"still_open":0.3,"closed_unmerged":0.0}
RANDOM_STATE=42
EMB_MODEL="all-MiniLM-L6-v2"
COST={"Copilot":1.10,"OpenAI_Codex":3.85,"Cursor":3.85,"Claude_Code":4.80,"Devin":4.50}
pd.set_option("display.width",140)

## 1. Load + clean title (strip agent identity)

In [2]:
df=pd.read_json(DATA_PATH,lines=True).reset_index(drop=True)
df["success"]=df["outcome"].map(SUCCESS_MAP)
AGENT_TOK=re.compile(r"\b(copilot|devin|cursor|codex|claude|openai|anthropic)\b",re.I)
FOOT=re.compile(r"generated with|generated by|co-authored-by|🤖",re.I)
def clean_title(t):
    t=FOOT.sub(" ",str(t)); t=AGENT_TOK.sub(" ",t); return re.sub(r"\s+"," ",t).strip()
df["title_clean"]=df["title"].map(clean_title)
# pre-treatment numeric/struct features only
df["log_stars"]=np.log1p(df["stars"]); df["log_forks"]=np.log1p(df["forks"])
df["has_issue_i"]=df["has_issue"].astype(int)
NUM=["log_stars","log_forks","has_issue_i"]                 # NOTE: no body-derived features
print("shape:",df.shape,"| dropped post-treatment: body, body_len, n_code, has_trace")
print(df.groupby("agent")["success"].agg(["mean","count"]).round(3))

shape: (25580, 21) | dropped post-treatment: body, body_len, n_code, has_trace
               mean  count
agent                     
Claude_Code   0.793   5116
Copilot       0.649   5116
Cursor        0.786   5116
Devin         0.662   5116
OpenAI_Codex  0.896   5116


## 2. Repo-grouped split (70/15/15)

In [3]:
def gsplit(d,ts):
    a,b=next(GroupShuffleSplit(1,test_size=ts,random_state=RANDOM_STATE).split(d,groups=d["repo_id"]))
    return d.iloc[a].copy(),d.iloc[b].copy()
train_df,temp=gsplit(df,0.30); val_df,test_df=gsplit(temp,0.50)
assert not(set(train_df.repo_id)&set(test_df.repo_id))
print(f"train {len(train_df)}  val {len(val_df)}  test {len(test_df)}")

train 18188  val 3497  test 3895


## 3. Encode cleaned title (frozen MiniLM, chunked)

In [4]:
def encode(texts,model_name):
    safe=model_name.replace("/","_"); cache=f"data/emb_title_{safe}_{len(texts)}.npy"
    if os.path.exists(cache): print("cached:",cache); return np.load(cache)
    import torch
    from sentence_transformers import SentenceTransformer
    dev="mps" if torch.backends.mps.is_available() else ("cuda" if torch.cuda.is_available() else "cpu")
    m=SentenceTransformer(model_name,device=dev)
    print(f"encoding {len(texts)} titles on {dev} ...")
    e=m.encode([t if str(t).strip() else " " for t in texts],batch_size=256,
               show_progress_bar=True,normalize_embeddings=True)
    np.save(cache,e); return e
EMB=encode(df["title_clean"],EMB_MODEL)
txt_tr,txt_va,txt_te=EMB[train_df.index.values],EMB[val_df.index.values],EMB[test_df.index.values]
print("title emb dim:",EMB.shape[1])

cached: data/emb_title_all-MiniLM-L6-v2_25580.npy
title emb dim: 384


## 4. Feature matrix (pre-treatment base + agent block)

In [5]:
lang_oh=OneHotEncoder(handle_unknown="infrequent_if_exist",min_frequency=50,sparse_output=False).fit(train_df[["language"]])
task_oh=OneHotEncoder(handle_unknown="ignore",sparse_output=False).fit(train_df[["task_type"]])
agent_oh=OneHotEncoder(categories=[AGENTS],handle_unknown="ignore",sparse_output=False).fit(train_df[["agent"]])
def base(d,txt,with_text=True):
    blocks=[d[NUM].to_numpy(float),lang_oh.transform(d[["language"]]),task_oh.transform(d[["task_type"]])]
    if with_text: blocks=[txt]+blocks
    return np.hstack(blocks)
def wa(b,ags): return np.hstack([b,agent_oh.transform(pd.DataFrame({"agent":ags}))])
bt,bv,bte=base(train_df,txt_tr),base(val_df,txt_va),base(test_df,txt_te)
Xtr,ytr=wa(bt,train_df.agent),train_df.success.to_numpy()
Xva,yva=wa(bv,val_df.agent),val_df.success.to_numpy()
Xte,yte=wa(bte,test_df.agent),test_df.success.to_numpy()
print("feature dim:",Xtr.shape[1],"= title",txt_tr.shape[1],"+ num",len(NUM),
      "+ lang/task one-hot + agent",len(AGENTS))

feature dim: 429 = title 384 + num 3 + lang/task one-hot + agent 5


## 5. Train XGBoost + routing scores

In [6]:
def fit_xgb(Xtr,ytr,Xva,yva):
    m=xgb.XGBRegressor(n_estimators=800,learning_rate=0.05,max_depth=6,subsample=0.8,
        colsample_bytree=0.8,min_child_weight=5,reg_lambda=1.0,early_stopping_rounds=40,
        random_state=RANDOM_STATE,n_jobs=-1)
    m.fit(Xtr,ytr,eval_set=[(Xva,yva)],verbose=False); return m
model=fit_xgb(Xtr,ytr,Xva,yva)
amean=train_df.groupby("agent").success.mean()
print(f"baseline(per-agent mean) test MAE {mean_absolute_error(yte,test_df.agent.map(amean)):.3f}")
print(f"v2 model              test MAE {mean_absolute_error(yte,model.predict(Xte)):.3f}")
scores=pd.DataFrame({a:model.predict(wa(bte,[a]*len(bte))) for a in AGENTS},index=test_df.index)

baseline(per-agent mean) test MAE 0.331
v2 model              test MAE 0.306


## 6. Routing success (validation-cell estimator, flat cost)

In [7]:
TOPL=set(train_df.language.value_counts().head(12).index)
def cell_of(d):
    lang=d.language.where(d.language.isin(TOPL),"Other"); return (lang+"|"+d.task_type).values
for d in (train_df,val_df,test_df): d["cell"]=cell_of(d)
def qtab(d):
    gm=d.groupby("agent").success.mean(); q=d.groupby(["cell","agent"]).success.mean().unstack().reindex(columns=AGENTS)
    for a in AGENTS: q[a]=q[a].fillna(gm[a])
    return q,gm
q_tr,gm_tr=qtab(train_df); q_va,gm_va=qtab(val_df)   # route by TRAIN cells; EVALUATE on VAL cells (independent)
tc=test_df.cell.values
def Q(tab,gm,agents): return np.array([tab.loc[c,a] if c in tab.index else gm[a] for c,a in zip(tc,agents)])
def pval(choice): return float(Q(q_va,gm_va,np.asarray(choice)).mean())   # independent val-cell estimator
def mcost(choice): return float(np.mean([COST[a] for a in choice]))

rng=np.random.default_rng(RANDOM_STATE); cell_best=q_tr.idxmax(1)
choices={
 "Always-Codex":      ["OpenAI_Codex"]*len(test_df),
 "Instance router":   list(scores[AGENTS].idxmax(1).values),
 "Cell argmax router": [cell_best.get(c,gm_tr.idxmax()) for c in tc],
 "Random":            list(rng.choice(AGENTS,len(test_df))),
}
print("success-only routing (validation-cell estimator, flat cost):")
for n,ch in choices.items():
    print(f"  {n:18} success={pval(ch):.3f}  cost=${mcost(ch):.2f}")
print("-> no learned router beats always-Codex on success alone; the cell signal pays off only under cost pressure (next cell).")

success-only routing (validation-cell estimator, flat cost):
  Always-Codex       success=0.880  cost=$3.85
  Instance router    success=0.876  cost=$3.92
  Cell argmax router success=0.862  cost=$3.99
  Random             success=0.759  cost=$3.65
-> no learned router beats always-Codex on success alone; the cell signal pays off only under cost pressure (next cell).


## 7. Cost-quality frontier: random vs instance vs cell router (new)

In [8]:
# Saving cost = sending a fraction of tasks to the only cheaper agent (Copilot). The question is WHICH.
valA={a:Q(q_va,gm_va,[a]*len(test_df)) for a in AGENTS}
qtrA={a:Q(q_tr,gm_tr,[a]*len(test_df)) for a in AGENTS}
gap_cell=qtrA["Copilot"]-qtrA["OpenAI_Codex"]      # NEW METHOD: per-(language x task_type) Copilot-Codex gap
gap_inst=scores["Copilot"].to_numpy()-scores["OpenAI_Codex"].to_numpy()
def frontier(score,label):
    o=np.argsort(-score); print(label)
    for p in [0.0,0.2,0.4,0.6,0.8]:
        k=int(round(p*len(test_df))); sel=np.zeros(len(test_df),bool); sel[o[:k]]=True
        succ=np.where(sel,valA["Copilot"],valA["OpenAI_Codex"]).mean()
        print(f"  frac->Copilot={p:.1f}  cost=${p*COST['Copilot']+(1-p)*COST['OpenAI_Codex']:.2f}  success={succ:.3f}")
frontier(rng.standard_normal(len(test_df)),"random mix:")
frontier(gap_inst,"instance router:")
frontier(gap_cell,"cell cost-gap router (new):")
print("-> at matched cost the cell router holds about 2 more points of success than random/instance.")

random mix:
  frac->Copilot=0.0  cost=$3.85  success=0.880
  frac->Copilot=0.2  cost=$3.30  success=0.841
  frac->Copilot=0.4  cost=$2.75  success=0.801
  frac->Copilot=0.6  cost=$2.20  success=0.761
  frac->Copilot=0.8  cost=$1.65  success=0.722
instance router:
  frac->Copilot=0.0  cost=$3.85  success=0.880
  frac->Copilot=0.2  cost=$3.30  success=0.839
  frac->Copilot=0.4  cost=$2.75  success=0.803
  frac->Copilot=0.6  cost=$2.20  success=0.763
  frac->Copilot=0.8  cost=$1.65  success=0.723
cell cost-gap router (new):
  frac->Copilot=0.0  cost=$3.85  success=0.880
  frac->Copilot=0.2  cost=$3.30  success=0.855
  frac->Copilot=0.4  cost=$2.75  success=0.821
  frac->Copilot=0.6  cost=$2.20  success=0.787
  frac->Copilot=0.8  cost=$1.65  success=0.737
-> at matched cost the cell router holds about 2 more points of success than random/instance.


## 8. Ablation: repo-metadata only (no text) -- strict lower bound

In [9]:
bt2,bv2,bte2=base(train_df,None,with_text=False),base(val_df,None,with_text=False),base(test_df,None,with_text=False)
m2=fit_xgb(wa(bt2,train_df.agent),ytr,wa(bv2,val_df.agent),yva)
sc2=pd.DataFrame({a:m2.predict(wa(bte2,[a]*len(bte2))) for a in AGENTS})
print("routing success (direct method):")
print(f"  Always-Codex            {pval(['OpenAI_Codex']*len(test_df)):.3f}")
print(f"  Router v2 (title+repo)   {pval(scores[AGENTS].idxmax(1).values):.3f}")
print(f"  Router v2 (repo-only)    {pval(sc2[AGENTS].idxmax(1).values):.3f}")
print(f"  v2 test MAE (repo-only)  {mean_absolute_error(yte,m2.predict(wa(bte2,test_df.agent))):.3f}")

routing success (direct method):
  Always-Codex            0.880
  Router v2 (title+repo)   0.876
  Router v2 (repo-only)    0.877
  v2 test MAE (repo-only)  0.309


## What this tests

If the v2 router (reduced-leakage features) still does NOT beat always-Codex on
success, that confirms the negative conclusion is not an artifact of leakage -- it holds under
honest, routing-time inputs. Cost-side value (if any) is the cost-quality frontier above.
Deferred (do later): time-based split, IPW/bootstrap CIs, still_open as binary/censored, and
the final write-up framing.